In [ ]:
# Install in VSCode terminal:
# pip install sentence-transformers
# pip install scikit-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(text: str):
    return model.encode([text])[0]

def rank_candidates(user_query: str, candidates: list) -> list:
    query_vec = embed(user_query)
    for c in candidates:
        desc = (f"{c['title']} {c['year']} | "
                f"Genres: {', '.join(c['genres'])} | "
                f"Rating: {c['rating']}/10 | "
                f"Cast: {', '.join(c['cast'])} | "
                f"Director: {c['director']} | "
                f"Overview: {c['overview']}")
        c["vector"] = embed(desc)
        c["score"] = float(cosine_similarity([query_vec],[c["vector"]])[0][0])
    return sorted(candidates, key=lambda x: x["score"], reverse=True)

In [ ]:
# Test 1 — embed function returns correct shape
vec = embed("I love action movies")
print(f"Test 1 — Embedding shape: {vec.shape}")
print(f"Expected: (384,) — {vec.shape == (384,)}")
print()

In [ ]:
# Test 2 — similar texts have high similarity
vec1 = embed("I love action movies with explosions")
vec2 = embed("I enjoy action films with fight scenes")
vec3 = embed("I like romantic comedies")

sim_high = float(cosine_similarity([vec1], [vec2])[0][0])
sim_low = float(cosine_similarity([vec1], [vec3])[0][0])
print(f"Test 2 — Similarity scores:")
print(f"Action vs Action: {sim_high:.3f} (should be high >0.7)")
print(f"Action vs Romance: {sim_low:.3f} (should be lower)")
print()

In [ ]:
# Test 3 — rank_candidates returns sorted results
candidates = [
    {"title": "John Wick", "year": "2014", "genres": ["Action", "Thriller"],
     "rating": 7.4, "cast": ["Keanu Reeves"], "director": "Chad Stahelski",
     "overview": "An ex-hitman comes out of retirement to track down the gangsters."},
    {"title": "Notting Hill", "year": "1999", "genres": ["Romance", "Comedy"],
     "rating": 7.1, "cast": ["Hugh Grant"], "director": "Roger Michell",
     "overview": "A British bookseller falls in love with an American actress."},
    {"title": "The Dark Knight", "year": "2008", "genres": ["Action", "Crime"],
     "rating": 9.0, "cast": ["Christian Bale"], "director": "Christopher Nolan",
     "overview": "Batman fights the Joker in Gotham City."},
]

query = "I love action movies with heroes"
ranked = rank_candidates(query, candidates)

print(f"Test 3 — Ranked results for '{query}':")
for i, m in enumerate(ranked, 1):
    print(f"{i}. {m['title']} — score: {m['score']:.3f}")
print()
print(f"Top result is action movie: {'Action' in ranked[0]['genres']}")